# Supplementary Figure 6 — historical limestone flows, 1931–2022

Historical **limestone** extraction, import, and export (for cement clinker production), and in-use **binder** stock during 1931–2022 for 142 economies in IMAGINE-Concrete, 30 economies per page (5 pages). Shaded areas indicate 95% confidence intervals (95% CI).

Per panel: grey = in-use binder stock (`S_binder_annual`, left axis); blue = extraction, green = export, red = import (`limestone_*`, right axis). The median line and 95% CI band are the per-year 50th / 2.3rd / 97.7th percentiles over the n_sim=10,000 historical Monte-Carlo runs (`MF/{economy}/historical/*.npz`).

> Run in the activated `wty` kernel: this env's numpy LAPACK crashes (`0xc06d007f`) if Python is started without the env activated.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / '1-IMAGINE-Concrete'))

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.legend_handler import HandlerBase
from tqdm import tqdm

from IMAGINE_Concrete import RESULTS
MF = RESULTS / 'material-flows'

In [2]:
ROOT    = Path('..')
DATA    = ROOT / 'data'
FIGURES = ROOT / 'figures'

economies_142 = pd.read_csv(DATA / 'production-142.csv', index_col=0).columns.tolist()

HIST = 'historical'                         # per-economy historical npz folder
HIST_START, HIST_END = 1931, 2022           # historical npz columns span 1931..2022 (92 years)
YEARS = list(range(HIST_START, HIST_END + 1))

# material -> in-use stock key (gravel<->coarse, sand<->fine, limestone<->binder)
STOCK_KEY = {'gravel': 'S_coarse_annual', 'sand': 'S_fine_annual', 'limestone': 'S_binder_annual'}
PCTL = [2.3, 50, 97.7]                       # 95% CI bounds + median (old xlsx sheets '2.3'/'50'/'97.7')

In [3]:
def draw_historical_for_a_country(material, country, ax1):
    # median / 2.3 / 97.7 percentiles per year from the n_sim=10,000 historical npz
    hh = STOCK_KEY[material]
    keys = [hh, f'{material}_extraction', f'{material}_export', f'{material}_import']
    med, lo, hi = {}, {}, {}
    for k in keys:
        p = np.percentile(np.load(MF / country / HIST / f'{k}.npz')['data'], PCTL, axis=0)
        lo[k], med[k], hi[k] = p[0], p[1], p[2]
    DF_country     = pd.DataFrame(med, index=YEARS)
    DF_country_025 = pd.DataFrame(lo,  index=YEARS)
    DF_country_975 = pd.DataFrame(hi,  index=YEARS)

    scale_dict = dict({'Gt': 1e-6, 'Mt': 1e-3, 'kt': 1})
    idx = np.argmax(DF_country_975[[f'{material}_extraction', f'{material}_export', f'{material}_import']].max())
    the_max_1_ = DF_country_975[[f'{material}_extraction', f'{material}_export', f'{material}_import']].max().max() / 1e6
    the_max_2_ = DF_country_975[[hh]].max().max() / 1e6

    if the_max_1_ <= .1:
        yaxis_scale_1 = 'Mt'
        if the_max_1_ <= 0.0001:
            yaxis_scale_1 = 'kt'
    else:
        yaxis_scale_1 = 'Gt'

    if the_max_2_ <= .1:
        yaxis_scale_2 = 'Mt'
        if the_max_2_ <= 0.001:
            yaxis_scale_2 = 'kt'
    else:
        yaxis_scale_2 = 'Gt'

    scale_factor_1 = scale_dict[yaxis_scale_1]
    scale_factor_2 = scale_dict[yaxis_scale_2]
    the_max_1 = the_max_1_ * 1e6 * scale_factor_1 * 2
    the_max_2 = the_max_2_ * 1e6 * scale_factor_2 * 1.1

    ax1_1 = ax1.twinx()
    (DF_country * scale_factor_2).plot(y=[hh], color='grey', label=['material stock'], legend=False, zorder=1, ax=ax1)
    (DF_country * scale_factor_1).plot(y=[f'{material}_extraction'], label=['extraction'], zorder=2, legend=False, ax=ax1_1)
    (DF_country * scale_factor_1).plot(y=[f'{material}_export', f'{material}_import'], color=['g', 'r'], zorder=2,  label=['export', 'import'], legend=False, ax=ax1_1)
    ax1.fill_between(np.linspace(HIST_START, HIST_END, len(YEARS)), (DF_country_025 * scale_factor_2)[hh], (DF_country_975 * scale_factor_2)[hh], color='lightgrey', alpha=0.7, lw=0, zorder=0)
    ax1_1.fill_between(np.linspace(HIST_START, HIST_END, len(YEARS)), (DF_country_025 * scale_factor_1)[f'{material}_extraction'], (DF_country_975 * scale_factor_1)[f'{material}_extraction'], alpha=0.3, zorder=0)
    ax1_1.fill_between(np.linspace(HIST_START, HIST_END, len(YEARS)), (DF_country_025 * scale_factor_1)[f'{material}_export'], (DF_country_975 * scale_factor_1)[f'{material}_export'], color='lightgreen', alpha=0.7, zorder=0)
    ax1_1.fill_between(np.linspace(HIST_START, HIST_END, len(YEARS)), (DF_country_025 * scale_factor_1)[f'{material}_import'], (DF_country_975 * scale_factor_1)[f'{material}_import'], color='pink', alpha=0.7, zorder=0)

    ax1.set_xlim(HIST_START, HIST_END)
    ax1.set_ylim(0, the_max_2)
    ax1_1.set_ylim(0, the_max_1)
    ax1.grid('on', ls='--')
    ax1.set_xticks([1931, 1960, 1990, 2022])
    ax1.set_ylabel(yaxis_scale_2, color='grey',)
    color_list = ['tab:blue', 'g', 'r']
    ax1_1.set_ylabel(f'{yaxis_scale_1} per year', color=color_list[idx])
    ax1.set_title(country)
    ax1.tick_params(axis='y', colors='dimgrey')
    ax1_1.tick_params(axis='y', colors=color_list[idx])

In [ ]:
# 1) 自定义 handler：可调“带厚度”和左右留白，保证线与带等长
class HandlerLineWithBand(HandlerBase):
    def __init__(self, band_scale=0.85, pad_frac=0.10, **kwargs):
        """
        band_scale:  阴影带在 legend 小框里的相对高度(0~1)
        pad_frac:    左右留白比例，避免贴边；同时保证线和带同样长度
        """
        super().__init__(**kwargs)
        self.band_scale = band_scale
        self.pad_frac = pad_frac

    def create_artists(self, legend, orig_handle,
                       xdescent, ydescent, width, height, fontsize, trans):
        line, band = orig_handle

        # 统一左右留白，线和带用同一长度
        x0 = xdescent + self.pad_frac * width
        x1 = xdescent + (1 - self.pad_frac) * width
        w  = x1 - x0

        # 带：竖直方向居中
        bh = self.band_scale * height
        by = ydescent + (height - bh) / 2.0
        band_art = Rectangle((x0, by), w, bh,
                             facecolor=band.get_facecolor(),
                             edgecolor='none',
                             alpha=band.get_alpha(),
                             transform=trans)

        # 线：与带等长；linecap 设为 'butt' 防止两端外伸
        line_art = Line2D([x0, x1], [ydescent + 0.5*height]*2,
                          color=line.get_color(),
                          linestyle=line.get_linestyle(),
                          linewidth=line.get_linewidth(),
                          solid_capstyle='butt',
                          transform=trans)
        return [band_art, line_art]


# 2) 小工具：生成“线+带”的句柄（颜色/线宽/alpha 与图中一致）
def make_line_band(color, lw=2.0, ls='-', alpha=0.30):
    line = Line2D([], [], color=color, lw=lw, ls=ls, solid_capstyle='butt')
    band = Rectangle((0, 0), 1, 1, facecolor=color, alpha=alpha)  # 只取颜色/alpha
    return (line, band)

# material stock（灰）
st_handle = make_line_band(color='0.4', lw=2.0, alpha=0.25)
# extraction（蓝）
ex_handle = make_line_band(color='#1f77b4', lw=2.0, alpha=0.35)
# import（红）——若无实际CI，仅作视觉匹配可用更浅 alpha
im_handle = make_line_band(color='r', lw=1.8, alpha=0.18)
# export（绿）
expt_handle = make_line_band(color='g', lw=1.8, alpha=0.18)

handles = [st_handle, ex_handle, im_handle, expt_handle]
labels  = ['in-use stock (95% UI)', 'extraction (95% UI)', 'import (95% UI)', 'export (95% UI)']

In [ ]:
# 5 pages (30 economies each) -> Supplementary Figure 6 (limestone)
material = 'limestone'
for j in tqdm(range(5), desc=material):
    fig = plt.figure(figsize=(12, 11), constrained_layout=True)
    plt.rcParams['font.family'] = 'Arial'
    for i, country in enumerate(economies_142[j * 30:(j + 1) * 30]):
        axx = plt.subplot(6, 5, i + 1)
        draw_historical_for_a_country(material, country, axx)
    fig.legend(handles, labels,
               handler_map={tuple: HandlerLineWithBand(band_scale=2)},
               loc='lower center', bbox_to_anchor=(0.5, -.05 if j != 4 else .07),
               ncol=4, frameon=False)

    out = FIGURES / f'Supplementary-Figure-6-part{j+1}.png'
    fig.savefig(out, dpi=600, bbox_inches='tight')
    plt.close(fig)

limestone: 100%|██████████| 5/5 [00:36<00:00,  7.23s/it]
